# Projeto de Python para Finanças

In [1]:
# instalação
%pip install yfinance pandas numpy plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Parte 1: Obter cotações e construção de carteira

In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

acoes = ['ITUB4.SA', 'PETR4.SA', 'VALE3.SA', 'IVVB11.SA']
indices = ['^BVSP', '^GSPC', 'BRL=X', 'GC=F']

data_inicio = datetime.now() - timedelta(days=365)
data_inicio = data_inicio.strftime('%Y-%m-%d')
data_fim = datetime.now().strftime('%Y-%m-%d')
print(data_inicio, data_fim)

def pegar_cotacoes(lista_tickers, data_inicio, data_fim):
    df = yf.download(lista_tickers, data_inicio, data_fim, auto_adjust=False) # auto_adjust -> ajuste do preço de fechamento, decontando o valor do dividendo
    df = df['Adj Close']
    df = df.ffill() # preenche as linhas vazias com o valor anterior
    df = df.dropna() # remove vazios
    return df

lista_tickers = acoes + indices
df_cotacoes = pegar_cotacoes(lista_tickers, data_inicio, data_fim)
display(df_cotacoes)

2025-03-26 2026-03-26


[*********************100%***********************]  8 of 8 completed


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2025-03-26,5.6984,3020.899902,28.522856,366.450012,33.574238,52.961666,132520.0,5712.200195
2025-03-27,5.7335,3060.199951,28.558407,366.149994,33.825661,53.383968,133149.0,5693.310059
2025-03-28,5.7393,3086.500000,28.167320,359.850006,33.610157,52.842323,131902.0,5580.939941
2025-03-31,5.7581,3122.800049,27.918446,358.700012,33.367710,52.052811,130260.0,5611.850098
2025-04-01,5.6984,3118.899902,27.925243,358.500000,33.493420,52.502647,131147.0,5633.069824
...,...,...,...,...,...,...,...,...
2026-03-20,5.2199,4570.399902,41.549999,389.450012,45.669998,75.550003,176219.0,6506.479980
2026-03-23,5.3119,4404.100098,42.779999,388.000000,46.029999,77.489998,181932.0,6581.000000
2026-03-24,5.2329,4399.299805,42.540001,387.559998,47.270000,78.099998,182509.0,6556.370117


In [3]:
# carteira em termos de valores financeiros
dic_carteira = {
    'ITUB4.SA': 5000,
    'VALE3.SA': 3000,
    'PETR4.SA': 4000,
    'IVVB11.SA': 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())

print(total_investido)

for ativo in dic_carteira:
    preco_incial = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_incial
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes

df_carteira['Total'] = df_carteira.sum(axis=1)

display(df_carteira)

18000


,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2025-03-26,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2025-03-27,5006.232032,3023.921202,4029.954262,5995.087707,18055.195204
2025-03-28,4937.675331,2993.239857,4004.279375,5891.936048,17827.130612
2025-03-31,4894.048097,2948.518117,3975.394502,5873.106840,17691.067555
2025-04-01,4895.239736,2973.999003,3990.371406,5869.831978,17729.442123
...,...,...,...,...,...
2026-03-20,7283.632394,4279.510556,5441.076388,6376.586152,23380.805490
2026-03-23,7499.248873,4389.401065,5483.966489,6352.844651,23725.461078
2026-03-24,7457.177724,4423.954393,5631.699008,6345.640354,23858.471478


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [4]:
df_cotacoes['SP500 (R$)'] = df_cotacoes['^GSPC'] * df_cotacoes['BRL=X']
df_cotacoes['OURO (R$)'] = df_cotacoes['GC=F'] * df_cotacoes['BRL=X']
df_cotacoes['Dólar'] = df_cotacoes['BRL=X']

df_cotacoes = df_cotacoes.drop(columns=['BRL=X', 'GC=F', '^GSPC'])
display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500 (R$),OURO (R$),Dólar
Date,,,,,,,,
2025-03-26,28.522856,366.450012,33.574238,52.961666,132520.0,32550.401711,17214.296066,5.6984
2025-03-27,28.558407,366.149994,33.825661,53.383968,133149.0,32642.593243,17545.656432,5.7335
2025-03-28,28.167320,359.850006,33.610157,52.842323,131902.0,32030.687345,17714.348753,5.7393
2025-03-31,27.918446,358.700012,33.367710,52.052811,130260.0,32313.594231,17981.395064,5.7581
2025-04-01,27.925243,358.500000,33.493420,52.502647,131147.0,32099.485202,17772.739268,5.6984
...,...,...,...,...,...,...,...,...
2026-03-20,41.549999,389.450012,45.669998,75.550003,176219.0,33963.175704,23857.031050,5.2199
2026-03-23,42.779999,388.000000,46.029999,77.489998,181932.0,34957.614814,23394.139920,5.3119
2026-03-24,42.540001,387.559998,47.270000,78.099998,182509.0,34308.830122,23021.096576,5.2329


In [5]:
def calcular_rentabilidade(df):
    retorno = df.iloc[-1] / df.iloc[0] - 1
    return retorno

display(calcular_rentabilidade(df_carteira))
display(calcular_rentabilidade(df_cotacoes))

ITUB4.SA     0.511069
VALE3.SA     0.502030
PETR4.SA     0.414775
IVVB11.SA    0.056897
Total        0.336773
dtype: float64

Ticker
ITUB4.SA      0.511069
IVVB11.SA     0.056897
PETR4.SA      0.414775
VALE3.SA      0.502030
^BVSP         0.399215
SP500 (R$)    0.060523
OURO (R$)     0.384105
Dólar        -0.081005
dtype: float64

In [6]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao['Carteira'] = df_carteira['Total']

display(df_comparacao)
print(calcular_rentabilidade(df_comparacao))

Ticker,^BVSP,SP500 (R$),OURO (R$),Dólar,Carteira
Date,,,,,
2025-03-26,132520.0,32550.401711,17214.296066,5.6984,18000.000000
2025-03-27,133149.0,32642.593243,17545.656432,5.7335,18055.195204
2025-03-28,131902.0,32030.687345,17714.348753,5.7393,17827.130612
2025-03-31,130260.0,32313.594231,17981.395064,5.7581,17691.067555
2025-04-01,131147.0,32099.485202,17772.739268,5.6984,17729.442123
...,...,...,...,...,...
2026-03-20,176219.0,33963.175704,23857.031050,5.2199,23380.805490
2026-03-23,181932.0,34957.614814,23394.139920,5.3119,23725.461078
2026-03-24,182509.0,34308.830122,23021.096576,5.2329,23858.471478


Ticker
^BVSP         0.399215
SP500 (R$)    0.060523
OURO (R$)     0.384105
Dólar        -0.081005
Carteira      0.336773
dtype: float64


In [7]:
df_comparacao = df_comparacao / df_comparacao.iloc[0] - 1
display(df_comparacao)

import plotly.express as px

grafico = px.line(df_comparacao, x=df_comparacao.index, y=df_comparacao.columns) 
grafico.update_layout(template='plotly_dark') #modo dark
grafico.show()

Ticker,^BVSP,SP500 (R$),OURO (R$),Dólar,Carteira
Date,,,,,
2025-03-26,0.000000,0.000000,0.000000,0.000000,0.000000
2025-03-27,0.004746,0.002832,0.019249,0.006160,0.003066
2025-03-28,-0.004663,-0.015966,0.029049,0.007177,-0.009604
2025-03-31,-0.017054,-0.007275,0.044562,0.010477,-0.017163
2025-04-01,-0.010361,-0.013853,0.032441,0.000000,-0.015031
...,...,...,...,...,...
2026-03-20,0.329754,0.043403,0.385885,-0.083971,0.298934
2026-03-23,0.372864,0.073953,0.358995,-0.067826,0.318081
2026-03-24,0.377219,0.054022,0.337324,-0.081690,0.325471


### Parte 3: Análise de Risco

In [8]:
# Correlação - indica se elas tendem a variar juntas
import plotly.express as px
import numpy as np

df_cotacoes['Carteira'] = df_carteira['Total']
tabela_rentabilidade_diaria = df_cotacoes / df_cotacoes.shift(1) # Shift(1) faz referencia a linha anterior
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()
tabela_correlacao = tabela_rentabilidade_diaria.corr()

grafico_correlacao = px.imshow(df_cotacoes.corr(), text_auto=True, color_continuous_scale='greens') 
grafico_correlacao.update_layout(template='plotly_dark') #modo dark
grafico_correlacao.show()

# Variancia dos retornos diarios do ativo (aplicando a função logaritima)
tabela_volatilidade = tabela_rentabilidade_diaria.std() * np.sqrt(252)# std calculo o Desvio padrão - 252 -> media de dias uteis no ano 
display(tabela_volatilidade)

Ticker
ITUB4.SA      0.217090
IVVB11.SA     0.155976
PETR4.SA      0.244088
VALE3.SA      0.244741
^BVSP         0.162471
SP500 (R$)    0.229155
OURO (R$)     0.296125
Dólar         0.123771
Carteira      0.127614
dtype: float64

### Parte 4: Análise Técnica e Indicadores

In [26]:
ticker = 'VALE3.SA'

import yfinance as yf
import plotly.graph_objects as go

df = yf.download(ticker, '2020-01-01', '2026-03-26', multi_level_index=False)
display(df)

# media movel de 50 dias
df['MM50'] = df['Close'].rolling(50).mean()

# media movel de 200 dias
df['MM200'] = df['Close'].rolling(200).mean()

grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=df.index, open=df['Open'], close=df['Close'], high=df['High'], low=df['Low'], name='Preço'))
# Serie da MM50
grafico.add_trace(go.Scatter(x=df.index, y=df['MM50'], name='MM50', line={'color': 'blue', 'width': 1}))
# Serie da MM200
grafico.add_trace(go.Scatter(x=df.index, y=df['MM200'], name='MM200', line={'color': 'yellow', 'width': 1}))
grafico.update_layout(template='plotly_dark')
grafico.show()



[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,30.140158,30.201180,29.818395,29.945990,17509700
2020-01-03,29.918257,30.234470,29.724091,29.779567,17284800
2020-01-06,29.740730,29.846134,29.485541,29.846134,32787800
2020-01-07,29.957088,30.062492,29.624233,29.679708,16326400
2020-01-08,29.962639,30.162353,29.746282,30.068045,15298500
...,...,...,...,...,...
2026-03-19,76.629997,76.709999,74.059998,74.169998,27262900
2026-03-20,75.550003,76.699997,74.709999,76.209999,44208900
2026-03-23,77.489998,78.419998,75.970001,76.559998,23384500
